In [ ]:
# ============================================================
# CNN-GCN-CAM-OD Framework for Ocular Disease Classification
# ============================================================

import os
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize

from imblearn.combine import SMOTETomek

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# ============================================================
# Dataset Class
# ============================================================

class ODIRDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return img, label

In [ ]:
# ============================================================
# Data Augmentation
# ============================================================

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [ ]:

# ============================================================
# Custom CNN (IILR Extraction)
# ============================================================

class CustomCNN(nn.Module):
    def __init__(self):
        super(CustomCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1))
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return x  # F_vector (IILR)


# ============================================================
# 2-Layer GCN
# ============================================================

class GCNLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super(GCNLayer, self).__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, X, A_hat):
        return F.relu(self.linear(torch.matmul(A_hat, X)))


class TwoLayerGCN(nn.Module):
    def __init__(self, in_features, hidden_dim, out_features):
        super(TwoLayerGCN, self).__init__()
        self.gcn1 = GCNLayer(in_features, hidden_dim)
        self.gcn2 = GCNLayer(hidden_dim, out_features)

    def forward(self, X, A_hat):
        X = self.gcn1(X, A_hat)
        X = self.gcn2(X, A_hat)
        return X


# ============================================================
# Channel Attention Module (CAM)
# ============================================================

class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=8):
        super(ChannelAttention, self).__init__()
        self.fc1 = nn.Linear(in_channels, in_channels // reduction)
        self.fc2 = nn.Linear(in_channels // reduction, in_channels)

    def forward(self, x):
        avg_pool = torch.mean(x, dim=0, keepdim=True)
        attn = F.relu(self.fc1(avg_pool))
        attn = torch.sigmoid(self.fc2(attn))
        return x * attn


# ============================================================
# Full CNN-GCN-CAM-OD Model
# ============================================================

class CNN_GCN_CAM_OD(nn.Module):
    def __init__(self, num_classes=8):
        super(CNN_GCN_CAM_OD, self).__init__()

        self.cnn = CustomCNN()
        self.gcn = TwoLayerGCN(128, 64, 128)
        self.cam = ChannelAttention(128)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        F_vector = self.cnn(x)
        A_hat = build_adjacency(F_vector)
        RARR = self.gcn(F_vector, A_hat)
        attended = self.cam(RARR)
        out = self.classifier(attended)
        return out


# ============================================================
# Adjacency Matrix
# ============================================================

def build_adjacency(features):
    similarity = torch.mm(features, features.t())
    norm = torch.norm(features, dim=1)
    A = similarity / (norm[:, None] * norm[None, :] + 1e-8)
    A = A + torch.eye(A.size(0)).to(device)
    D = torch.diag(torch.pow(A.sum(1), -0.5))
    A_hat = torch.mm(torch.mm(D, A), D)
    return A_hat

In [ ]:
# ============================================================
# Plot Functions
# ============================================================

def plot_training_curves(train_losses, val_losses,
                         train_accs, val_accs):

    plt.figure()
    plt.plot(train_losses)
    plt.plot(val_losses)
    plt.title("Loss Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend(["Train", "Validation"])
    plt.show()

    plt.figure()
    plt.plot(train_accs)
    plt.plot(val_accs)
    plt.title("Accuracy Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend(["Train", "Validation"])
    plt.show()


def plot_confusion_matrix(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure()
    sns.heatmap(cm, annot=True, fmt="d")
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()


def plot_roc_curves(y_true, y_probs, num_classes):
    y_true_bin = label_binarize(y_true, classes=list(range(num_classes)))
    y_probs = np.array(y_probs)

    plt.figure()
    for i in range(num_classes):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"Class {i} (AUC={roc_auc:.2f})")

    plt.plot([0,1], [0,1], linestyle="--")
    plt.title("Multi-Class ROC")
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.legend()
    plt.show()


# ============================================================
# Training Function (4-Fold CV)
# ============================================================

def train_model(image_paths, labels, epochs=30):

    skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

    for fold, (train_idx, val_idx) in enumerate(skf.split(image_paths, labels)):
        print(f"\n========== Fold {fold+1} ==========")

        train_dataset = ODIRDataset(
            [image_paths[i] for i in train_idx],
            [labels[i] for i in train_idx],
            train_transform)

        val_dataset = ODIRDataset(
            [image_paths[i] for i in val_idx],
            [labels[i] for i in val_idx],
            val_transform)

        train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=16)

        model = CNN_GCN_CAM_OD().to(device)
        optimizer = optim.Adam(model.parameters(), lr=1e-4)
        criterion = nn.CrossEntropyLoss()

        train_losses, val_losses = [], []
        train_accs, val_accs = [], []

        for epoch in range(epochs):

            model.train()
            running_loss, correct, total = 0, 0, 0

            for imgs, lbls in train_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)

                outputs = model(imgs)
                loss = criterion(outputs, lbls)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                running_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                total += lbls.size(0)
                correct += (predicted == lbls).sum().item()

            train_losses.append(running_loss / len(train_loader))
            train_accs.append(correct / total)

            model.eval()
            val_running_loss, correct, total = 0, 0, 0
            all_preds, all_labels, all_probs = [], [], []

            with torch.no_grad():
                for imgs, lbls in val_loader:
                    imgs, lbls = imgs.to(device), lbls.to(device)
                    outputs = model(imgs)
                    loss = criterion(outputs, lbls)

                    val_running_loss += loss.item()
                    probs = torch.softmax(outputs, dim=1)
                    _, predicted = torch.max(outputs, 1)

                    total += lbls.size(0)
                    correct += (predicted == lbls).sum().item()

                    all_preds.extend(predicted.cpu().numpy())
                    all_labels.extend(lbls.cpu().numpy())
                    all_probs.extend(probs.cpu().numpy())

            val_losses.append(val_running_loss / len(val_loader))
            val_accs.append(correct / total)

            print(f"Epoch {epoch+1}/{epochs} "
                  f"Train Acc: {train_accs[-1]:.4f} "
                  f"Val Acc: {val_accs[-1]:.4f}")

        # Plot results
        plot_training_curves(train_losses, val_losses,
                             train_accs, val_accs)

        plot_confusion_matrix(all_labels, all_preds)
        plot_roc_curves(all_labels, all_probs, 8)

        print(classification_report(all_labels, all_preds))


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    # Example loading (modify path)
    df = pd.read_csv("odir_labels.csv")
    image_paths = df["image_path"].values
    labels = df["label"].values

    train_model(image_paths, labels, epochs=30)